In [1]:
!pip install rasterio numpy 
# install the libraries
# don't need the ! outside of jupyter

In [2]:
import rasterio
import numpy as np
import os
import pandas as pd
import glob
#import libraries

In [3]:
raster_folder = r"C:\Users\svbai\OneDrive\Desktop\flood_mask_rasters" # path to flood mask raster folder, projected in British National Grid
raster_list = glob.glob(os.path.join(raster_folder, "*.tif"))

print("Rasters found:")
for raster in raster_list:
    print(raster)

Rasters found:
C:\Users\svbai\OneDrive\Desktop\flood_mask_rasters\flood_mask_2021_BNG.tif
C:\Users\svbai\OneDrive\Desktop\flood_mask_rasters\flood_mask_2122_BNG.tif
C:\Users\svbai\OneDrive\Desktop\flood_mask_rasters\flood_mask_2223_BNG.tif
C:\Users\svbai\OneDrive\Desktop\flood_mask_rasters\flood_mask_2324.tif
C:\Users\svbai\OneDrive\Desktop\flood_mask_rasters\flood_mask_2324_BNG.tif
C:\Users\svbai\OneDrive\Desktop\flood_mask_rasters\flood_mask_2425_BNG.tif


In [12]:
def analyse_flood_raster(raster_path, flood_value=1):
    with rasterio.open(raster_path) as src:
        raster = src.read(1) # read the first band
        nodata = src.nodata # get NoData value if there is one
        crs = src.crs # identify CRS of raster
        warning_message = ""
    
        pixel_width = src.transform.a # get pixel size
        pixel_height = abs(src.transform.e)
        pixel_area = pixel_width * pixel_height

        if crs and crs.is_geographic:
            warning_message = (
                "Raster is in a geographic CRS (degrees). "
                "Area calculation in m2 / km2 may not be valid."
            )
            
            print(
                f"\033[91mWARNING for {os.path.basename(raster_path)}: "
                f"{warning_message}\033[0m"
            )

        if nodata is not None: # create a valid data mask
            valid_mask = raster != nodata
        else:
            valid_mask = np.ones(raster.shape, dtype=bool)

        flood_pixels = np.sum((raster == flood_value) & valid_mask) # count flooded pixels
        total_valid_pixels = np.sum(valid_mask) # count valid pixels
        flooded_area_m2 = flood_pixels * pixel_area # calculate flooded area
        flooded_area_km2 = flooded_area_m2 / 1_000_000

        if total_valid_pixels > 0:
            percent_flooded = (flood_pixels / total_valid_pixels) * 100
        else:
            percent_flooded = 0

        results = {
            "Raster": os.path.basename(raster_path),
            "CRS": str(crs),
            "Warning": warning_message,
            "Flooded Pixels": flood_pixels,
            "Total Valid Pixels": total_valid_pixels,
            "Pixel Area m2": pixel_area,
            "Flooded Area m2": flooded_area_m2,
            "Flooded Area km2": flooded_area_km2,
            "Percent Flooded": percent_flooded
        }

    return results

## Loop rasters through function

In [13]:
all_results = []

for raster_path in raster_list:
    print(f"Processing: {os.path.basename(raster_path)}")
    result = analyse_flood_raster(raster_path, flood_value = 1)
    all_results.append(result)

df = pd.DataFrame(all_results)

print(df)
print(f"Number of rows in dataframe: {len(df)}")

Processing: flood_mask_2021_BNG.tif
Processing: flood_mask_2122_BNG.tif
Processing: flood_mask_2223_BNG.tif
Processing: flood_mask_2324.tif
WARNING for flood_mask_2324.tif: Raster is in a geographic CRS (degrees). Area calculation in m2 / km2 may not be valid.
Processing: flood_mask_2324_BNG.tif
Processing: flood_mask_2425_BNG.tif
                    Raster         CRS  \
0  flood_mask_2021_BNG.tif  EPSG:27700   
1  flood_mask_2122_BNG.tif  EPSG:27700   
2  flood_mask_2223_BNG.tif  EPSG:27700   
3      flood_mask_2324.tif   EPSG:4326   
4  flood_mask_2324_BNG.tif  EPSG:27700   
5  flood_mask_2425_BNG.tif  EPSG:27700   

                                             Warning  Flooded Pixels  \
0                                                              34449   
1                                                              34449   
2                                                               2454   
3  Raster is in a geographic CRS (degrees). Area ...           49384   
4           

## Export Results to a CSV

In [14]:
os.makedirs(r"C:\Users\svbai\B01051807_Bain_EGM722_Programming_Project\outputs", exist_ok=True) # create outputs folder if it doesn't already exist

df.to_csv(r"C:\Users\svbai\B01051807_Bain_EGM722_Programming_Project\outputs\flood_summary_all.csv", index=False)

print(df)
print("CSV saved to outputs/flood_summary_all.csv")

                    Raster         CRS  \
0  flood_mask_2021_BNG.tif  EPSG:27700   
1  flood_mask_2122_BNG.tif  EPSG:27700   
2  flood_mask_2223_BNG.tif  EPSG:27700   
3      flood_mask_2324.tif   EPSG:4326   
4  flood_mask_2324_BNG.tif  EPSG:27700   
5  flood_mask_2425_BNG.tif  EPSG:27700   

                                             Warning  Flooded Pixels  \
0                                                              34449   
1                                                              34449   
2                                                               2454   
3  Raster is in a geographic CRS (degrees). Area ...           49384   
4                                                              51877   
5                                                               3126   

   Total Valid Pixels  Pixel Area m2  Flooded Area m2  Flooded Area km2  \
0             1966152   7.010250e+02     2.414961e+07      2.414961e+01   
1             1966152   7.010250e+02     2.414961e